# Recipe Rating Classification with Recipe and Ingredient Features

This notebook predicts whether a recipe belongs to the **lower-rated** group.

The model uses two types of features:

1. **Recipe-level numerical features**
   - calories
   - carbs
   - fat
   - protein
   - ingredient count
   - rating count

2. **Ingredient features**
   - `contains_sugar`
   - `contains_chicken`
   - `contains_butter`
   - etc.

## Classification target

The threshold is editable at the beginning of the notebook.

```python
is_lower_rated = 1 if rating_value < selected_threshold
is_lower_rated = 0 if rating_value >= selected_threshold
```

Default value:

```python
selected_threshold = 4.5
```


## 1. Choose the Rating Threshold

In [21]:
# ============================================================
# USER PARAMETER
# Change this value if you want another split.
# Suggested values: 4.0, 4.2, 4.3, 4.4, 4.5
# ============================================================

selected_threshold = 4.4


## 2. Import Libraries and Load Data

In [22]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import re

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

allrecipes_df = pd.read_csv("allrecipes_all.csv")
ingredients_df = pd.read_csv("recipes_ingredients_long.csv")

print("AllRecipes shape:", allrecipes_df.shape)
print("Ingredients shape:", ingredients_df.shape)
print("Selected threshold:", selected_threshold)


AllRecipes shape: (14438, 25)
Ingredients shape: (142466, 13)
Selected threshold: 4.4


## 3. Helper Functions

In [23]:
def extract_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value).replace(",", "")
    match = re.search(r"[-+]?\d*\.?\d+", value)

    if match:
        return float(match.group())
    return np.nan


def evaluate_model(model_name, y_true, y_pred, y_proba):
    return {
        "Model": model_name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_true, y_proba)
    }


## 4. Build Recipe-Level Dataset

The ingredient dataset has multiple rows per recipe.

We aggregate it to one row per recipe and create these numerical features:

- `calories`
- `carbs`
- `fat`
- `protein`
- `ingredient_count`
- `rating_count`
- `log_rating_count`

`log_rating_count` is useful because `rating_count` is usually very skewed.


In [24]:
# Convert nutrition columns to numeric
for col in ["nutrition_calories", "nutrition_carbs", "nutrition_fat", "nutrition_protein"]:
    ingredients_df[col + "_num"] = ingredients_df[col].apply(extract_number)


recipe_summary = (
    ingredients_df
    .groupby(["recipe_id", "title", "url"], as_index=False)
    .agg(
        ingredient_count=("ingredient_canonical", "nunique"),
        rating_value=("rating_value", "first"),
        rating_count=("rating_count", "first"),
        calories=("nutrition_calories_num", "first"),
        carbs=("nutrition_carbs_num", "first"),
        fat=("nutrition_fat_num", "first"),
        protein=("nutrition_protein_num", "first")
    )
)

recipe_summary = recipe_summary.dropna(subset=["rating_value", "rating_count"]).copy()

# Optional: remove recipes with no ratings
recipe_summary = recipe_summary[recipe_summary["rating_count"] > 0].copy()

recipe_summary["log_rating_count"] = np.log1p(recipe_summary["rating_count"])

print("Recipe-level dataset shape:", recipe_summary.shape)
print(recipe_summary[["rating_value", "rating_count", "ingredient_count", "calories", "carbs", "fat", "protein"]].describe())


Recipe-level dataset shape: (13516, 11)
       rating_value  rating_count  ingredient_count      calories  \
count  13516.000000  13516.000000      13516.000000  13334.000000   
mean       4.529609    223.771826          9.642646    347.945328   
std        0.411447    773.943961          3.938446    241.403873   
min        1.000000      1.000000          1.000000      1.000000   
25%        4.400000      6.000000          7.000000    183.000000   
50%        4.600000     32.000000          9.000000    310.000000   
75%        4.800000    148.000000         12.000000    457.000000   
max        5.000000  20956.000000         32.000000   5109.000000   

              carbs           fat       protein  
count  13320.000000  13184.000000  13289.000000  
mean      33.008258     18.044448     14.647227  
std       28.167366     15.865053     16.084600  
min        0.000000      0.000000      0.000000  
25%       14.000000      7.000000      4.000000  
50%       29.000000     14.000000     

## 5. Compare Possible Rating Thresholds

In [25]:
threshold_options = [4.0, 4.2, 4.3, 4.4, 4.5, selected_threshold]
threshold_options = sorted(set(threshold_options))

threshold_summary = []

for threshold in threshold_options:
    lower_count = (recipe_summary["rating_value"] < threshold).sum()
    total_count = len(recipe_summary)
    lower_share = lower_count / total_count

    threshold_summary.append({
        "threshold": threshold,
        "lower_rated_count": lower_count,
        "total_recipes": total_count,
        "lower_rated_share": lower_share
    })

threshold_summary_df = pd.DataFrame(threshold_summary)

print(threshold_summary_df)

fig = px.bar(
    threshold_summary_df,
    x="threshold",
    y="lower_rated_share",
    text=threshold_summary_df["lower_rated_share"].round(3),
    title=f"Share of Lower-Rated Recipes by Threshold — Selected threshold = {selected_threshold}",
    labels={
        "threshold": "Rating threshold",
        "lower_rated_share": "Share of recipes below threshold"
    }
)

fig.add_vline(
    x=selected_threshold,
    line_dash="dash",
    annotation_text=f"Selected threshold = {selected_threshold}",
    annotation_position="top right"
)

fig.show()


   threshold  lower_rated_count  total_recipes  lower_rated_share
0        4.0                823          13516           0.060891
1        4.2               1813          13516           0.134137
2        4.3               2269          13516           0.167875
3        4.4               3040          13516           0.224919
4        4.5               4082          13516           0.302012


## 6. Define the Classification Target

In [26]:
recipe_summary["is_lower_rated"] = (
    recipe_summary["rating_value"] < selected_threshold
).astype(int)

print("Target definition:")
print(f"0 = rating_value >= {selected_threshold}")
print(f"1 = rating_value < {selected_threshold}")

print("\nTarget distribution:")
print(recipe_summary["is_lower_rated"].value_counts())

print("\nTarget distribution (%):")
print((recipe_summary["is_lower_rated"].value_counts(normalize=True) * 100).round(2))


Target definition:
0 = rating_value >= 4.4
1 = rating_value < 4.4

Target distribution:
is_lower_rated
0    10476
1     3040
Name: count, dtype: int64

Target distribution (%):
is_lower_rated
0    77.51
1    22.49
Name: proportion, dtype: float64


In [27]:
fig = px.histogram(
    recipe_summary,
    x="rating_value",
    nbins=30,
    title=f"Distribution of Ratings with Selected Threshold = {selected_threshold}",
    labels={"rating_value": "Rating value"}
)

fig.add_vline(
    x=selected_threshold,
    line_dash="dash",
    annotation_text=f"Selected threshold = {selected_threshold}",
    annotation_position="top left"
)

fig.show()


In [28]:
target_counts = (
    recipe_summary["is_lower_rated"]
    .value_counts()
    .rename_axis("target")
    .reset_index(name="count")
)

target_counts["label"] = target_counts["target"].map({
    0: f"Not lower-rated\n(rating >= {selected_threshold})",
    1: f"Lower-rated\n(rating < {selected_threshold})"
})

fig = px.bar(
    target_counts,
    x="label",
    y="count",
    text="count",
    title=f"Classification Target Distribution — Threshold: rating_value < {selected_threshold}",
    labels={
        "label": "Class",
        "count": "Number of recipes"
    }
)

fig.show()


## 7. Create Ingredient Features

We keep the top 100 most frequent ingredients and convert them into binary variables.

Example:

```python
contains_sugar = 1
contains_chicken = 0
```


In [29]:
top_n_ingredients = 100

top_ingredients = (
    ingredients_df["ingredient_canonical"]
    .dropna()
    .value_counts()
    .head(top_n_ingredients)
    .index
)

ingredient_subset = ingredients_df[
    ingredients_df["ingredient_canonical"].isin(top_ingredients)
].copy()

ingredient_features = (
    pd.crosstab(
        ingredient_subset["recipe_id"],
        ingredient_subset["ingredient_canonical"]
    )
    .clip(upper=1)
)

ingredient_features.columns = [
    "contains_" + str(col).replace(" ", "_").replace("-", "_")
    for col in ingredient_features.columns
]

ingredient_features = ingredient_features.reset_index()

print("Ingredient feature matrix shape:", ingredient_features.shape)
print("Number of ingredient features:", len(ingredient_features.columns) - 1)


Ingredient feature matrix shape: (13756, 101)
Number of ingredient features: 100


## 8. Final Modeling Dataset

This version uses both:

- recipe-level numerical features
- ingredient binary features


In [30]:
model_df = recipe_summary.merge(
    ingredient_features,
    on="recipe_id",
    how="left"
)

ingredient_feature_cols = [
    col for col in model_df.columns if col.startswith("contains_")
]

model_df[ingredient_feature_cols] = model_df[ingredient_feature_cols].fillna(0)

recipe_feature_cols = [
    "calories",
    "carbs",
    "fat",
    "protein",
    "ingredient_count",
    "rating_count",
    "log_rating_count"
]

feature_cols = recipe_feature_cols + ingredient_feature_cols

X = model_df[feature_cols]
y = model_df["is_lower_rated"]

print("Final modeling dataset shape:", model_df.shape)
print("Recipe-level features:", recipe_feature_cols)
print("Number of ingredient features:", len(ingredient_feature_cols))
print("Total number of features:", len(feature_cols))


Final modeling dataset shape: (13516, 112)
Recipe-level features: ['calories', 'carbs', 'fat', 'protein', 'ingredient_count', 'rating_count', 'log_rating_count']
Number of ingredient features: 100
Total number of features: 107


## 9. Train-Test Split

In [31]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

print("\nTraining target distribution (%):")
print((y_train.value_counts(normalize=True) * 100).round(2))

print("\nTesting target distribution (%):")
print((y_test.value_counts(normalize=True) * 100).round(2))


Training rows: 10812
Testing rows: 2704

Training target distribution (%):
is_lower_rated
0    77.51
1    22.49
Name: proportion, dtype: float64

Testing target distribution (%):
is_lower_rated
0    77.51
1    22.49
Name: proportion, dtype: float64


## 10. Train Classification Models

We train:

1. Logistic Regression
2. Random Forest

Logistic Regression uses scaling and median imputation.

Random Forest uses median imputation and handles non-linear relationships.


In [32]:
logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_test)
logistic_proba = logistic_model.predict_proba(X_test)[:, 1]

logistic_results = evaluate_model(
    "Logistic Regression",
    y_test,
    logistic_pred,
    logistic_proba
)

print(logistic_results)


{'Model': 'Logistic Regression', 'Accuracy': 0.613905325443787, 'Precision': 0.3244766505636071, 'Recall': 0.662828947368421, 'F1': 0.43567567567567567, 'ROC_AUC': 0.6786571853656087}


In [33]:
rf_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=12,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]

rf_results = evaluate_model(
    "Random Forest",
    y_test,
    rf_pred,
    rf_proba
)

print(rf_results)


{'Model': 'Random Forest', 'Accuracy': 0.6142751479289941, 'Precision': 0.3176865046102263, 'Recall': 0.6233552631578947, 'F1': 0.42087729039422545, 'ROC_AUC': 0.667611710275211}


## 11. Model Performance

In [34]:
results_df = pd.DataFrame([
    logistic_results,
    rf_results
])

print(results_df)

results_plot_df = results_df.melt(
    id_vars="Model",
    value_vars=["Accuracy", "Precision", "Recall", "F1", "ROC_AUC"],
    var_name="Metric",
    value_name="Value"
)

fig = px.bar(
    results_plot_df,
    x="Model",
    y="Value",
    color="Metric",
    barmode="group",
    title=f"Classification Performance — Threshold = {selected_threshold}"
)

fig.show()


                 Model  Accuracy  Precision    Recall        F1   ROC_AUC
0  Logistic Regression  0.613905   0.324477  0.662829  0.435676  0.678657
1        Random Forest  0.614275   0.317687  0.623355  0.420877  0.667612


## 12. ROC AUC Curve

In [35]:
log_fpr, log_tpr, _ = roc_curve(y_test, logistic_proba)
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_proba)

log_auc = roc_auc_score(y_test, logistic_proba)
rf_auc = roc_auc_score(y_test, rf_proba)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=log_fpr,
    y=log_tpr,
    mode="lines",
    name=f"Logistic Regression AUC = {log_auc:.3f}"
))

fig.add_trace(go.Scatter(
    x=rf_fpr,
    y=rf_tpr,
    mode="lines",
    name=f"Random Forest AUC = {rf_auc:.3f}"
))

fig.add_trace(go.Scatter(
    x=[0, 1],
    y=[0, 1],
    mode="lines",
    name="Random classifier",
    line=dict(dash="dash")
))

fig.update_layout(
    title=f"ROC AUC Curve — Threshold = {selected_threshold}",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    width=800,
    height=550
)

fig.show()


## 13. Confusion Matrices

In [36]:
log_cm = confusion_matrix(y_test, logistic_pred)
rf_cm = confusion_matrix(y_test, rf_pred)

log_cm_df = pd.DataFrame(
    log_cm,
    index=["Actual not lower-rated", "Actual lower-rated"],
    columns=["Predicted not lower-rated", "Predicted lower-rated"]
)

rf_cm_df = pd.DataFrame(
    rf_cm,
    index=["Actual not lower-rated", "Actual lower-rated"],
    columns=["Predicted not lower-rated", "Predicted lower-rated"]
)

fig = px.imshow(
    log_cm_df,
    text_auto=True,
    title=f"Confusion Matrix — Logistic Regression — Threshold = {selected_threshold}"
)
fig.show()

fig = px.imshow(
    rf_cm_df,
    text_auto=True,
    title=f"Confusion Matrix — Random Forest — Threshold = {selected_threshold}"
)
fig.show()


## 14. Random Forest Feature Importance

This plot now includes the kind of features you requested:

- calories
- rating count
- fat
- ingredient count
- protein
- carbs
- ingredient indicators

These are useful for understanding which variables the Random Forest uses most.


In [37]:
rf_classifier = rf_model.named_steps["model"]

feature_importance_df = pd.DataFrame({
    "Feature": feature_cols,
    "Importance": rf_classifier.feature_importances_
}).sort_values("Importance", ascending=False)

print("Top 30 Random Forest feature importances:")
print(feature_importance_df.head(30))

fig = px.bar(
    feature_importance_df.head(30),
    x="Feature",
    y="Importance",
    title=f"Top 30 Random Forest Feature Importances — Threshold = {selected_threshold}"
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()


Top 30 Random Forest feature importances:
                             Feature  Importance
5                       rating_count    0.215405
6                   log_rating_count    0.212929
2                                fat    0.063188
0                           calories    0.061242
1                              carbs    0.054528
4                   ingredient_count    0.051005
3                            protein    0.050948
84                     contains_salt    0.008937
65                     contains_milk    0.008653
55              contains_kosher_salt    0.008260
40                      contains_egg    0.007931
103                   contains_water    0.007908
45                   contains_garlic    0.007220
8         contains_all_purpose_flour    0.007097
69                contains_olive_oil    0.006866
50              contains_heavy_cream    0.006633
27                 contains_cinnamon    0.006430
17                   contains_butter    0.006425
56                contains_

## 15. Features Associated with Lower Ratings

Random Forest importance shows which variables matter, but it does not show whether they increase or decrease the probability of a lower rating.

For direction, we use Logistic Regression coefficients.

A positive coefficient means:

```python
feature increases probability of is_lower_rated = 1
```

So the plot below shows features most associated with lower ratings.


In [38]:
logistic_classifier = logistic_model.named_steps["model"]

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": logistic_classifier.coef_[0]
})

lowering_features_df = (
    coef_df
    .sort_values("Coefficient", ascending=False)
    .head(30)
)

print("Top 30 features associated with lower ratings:")
print(lowering_features_df)

fig = px.bar(
    lowering_features_df,
    x="Feature",
    y="Coefficient",
    title=f"Features Most Associated with Lower Ratings — Logistic Regression — Threshold = {selected_threshold}",
    labels={
        "Coefficient": "Positive coefficient: increases probability of lower-rated class"
    }
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()


Top 30 features associated with lower ratings:
                               Feature  Coefficient
87            contains_salt_and_pepper     0.131464
3                              protein     0.118540
103                     contains_water     0.117798
65                       contains_milk     0.108774
70                      contains_onion     0.101607
106      contains_worcestershire_sauce     0.092891
84                       contains_salt     0.088631
1                                carbs     0.083664
100           contains_vanilla_extract     0.076160
8           contains_all_purpose_flour     0.072528
12              contains_baking_powder     0.072188
40                        contains_egg     0.070558
94                  contains_soy_sauce     0.069926
92                contains_small_onion     0.067017
62                  contains_margarine     0.053270
99   contains_unsweetened_cocoa_powder     0.052976
76                    contains_parsley     0.051625
61               

## 16. Ingredients with Highest Lower-Rated Share

This descriptive plot focuses only on ingredient features.

It shows the ingredients whose recipes have the highest share of lower-rated cases.


In [39]:
ingredient_signal_rows = []

for col in ingredient_feature_cols:
    recipes_with_ingredient = model_df[model_df[col] == 1]

    if len(recipes_with_ingredient) >= 30:
        ingredient_signal_rows.append({
            "Ingredient feature": col,
            "Recipe count": len(recipes_with_ingredient),
            "Average rating": recipes_with_ingredient["rating_value"].mean(),
            "Lower-rated share": recipes_with_ingredient["is_lower_rated"].mean()
        })

ingredient_signal_df = pd.DataFrame(ingredient_signal_rows)

lowering_share_df = (
    ingredient_signal_df
    .sort_values("Lower-rated share", ascending=False)
    .head(20)
)

print("Top 20 ingredients by lower-rated share:")
print(lowering_share_df)

fig = px.bar(
    lowering_share_df,
    x="Ingredient feature",
    y="Lower-rated share",
    hover_data=["Recipe count", "Average rating"],
    title=f"Ingredients with Highest Lower-Rated Share — Threshold = {selected_threshold}",
    labels={
        "Lower-rated share": f"Share of recipes with rating < {selected_threshold}"
    }
)

fig.update_layout(xaxis_tickangle=-45)
fig.show()


Top 20 ingredients by lower-rated share:
                   Ingredient feature  Recipe count  Average rating  \
34                 contains_egg_white           187        4.419251   
92  contains_unsweetened_cocoa_powder           211        4.462085   
98                contains_whole_milk           178        4.439888   
3             contains_almond_extract           188        4.486170   
35                  contains_egg_yolk           234        4.475641   
55                 contains_margarine           195        4.492821   
84                contains_shortening           195        4.461026   
80           contains_salt_and_pepper           612        4.475490   
58                      contains_milk          1417        4.471277   
33                       contains_egg          2114        4.476821   
61                    contains_nutmeg           649        4.493374   
85               contains_small_onion           259        4.493050   
5              contains_baking_powde

## Conclusion

This version uses richer model features:

- calories
- carbs
- fat
- protein
- ingredient count
- rating count
- log rating count
- top ingredient indicators

The Random Forest feature importance plot now shows which recipe-level and ingredient-level variables are most useful for classification.

The Logistic Regression coefficient plot shows which features are associated with lower ratings.
